══════════════════════════════════════════════════════════════
 WEEK 8  |  DAY 3  |  DATA QUALITY
══════════════════════════════════════════════════════════════

 LEARNING OBJECTIVES
 ───────────────────
 1. Define the four core data quality dimensions
 2. Write reusable quality check functions in Python
 3. Build a quality report that runs all checks and summarizes results

 TIME:  ~30 minutes

 YOUTUBE
 ───────
 Search: "data quality checks Python pandas tutorial"
 Search: "data quality framework completeness accuracy"

══════════════════════════════════════════════════════════════


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# ─────────────────────────────────────────────────────────────
# ARCHITECTURE DECISION
# ─────────────────────
# Choosing between: manual assert statements vs Pydantic schema validation vs Great Expectations.
# Rule of thumb: use assert for quick in-pipeline checks. Move to Great Expectations when you need
# shareable, version-controlled data contracts across teams.

Sample dataset -- realistic sales pipeline output with intentional quality issues


In [ ]:
np.random.seed(42)
n = 200

records = {
    "sale_id":      list(range(1, 201)),
    "sale_date":    pd.date_range("2024-01-01", periods=n, freq="1D").strftime("%Y-%m-%d").tolist(),
    "rep_name":     np.random.choice(["Alice", "Bob", "Carol", "Dave", "Eve", None], n, p=[0.25,0.25,0.20,0.15,0.10,0.05]).tolist(),
    "region":       np.random.choice(["West", "East", "Central", "INVALID_REGION", None], n, p=[0.35,0.30,0.25,0.07,0.03]).tolist(),
    "product":      np.random.choice(["Laptop", "Monitor", "Keyboard", "Headset", "Mouse"], n).tolist(),
    "quantity":     (np.random.randint(1, 20, n) * np.random.choice([1, 1, 1, -1], n, p=[0.9,0.05,0.02,0.03])).tolist(),
    "unit_price":   np.round(np.abs(np.random.normal(300, 200, n)), 2).tolist(),
    "revenue":      None,   # will be computed below (with some intentional errors)
}

df = pd.DataFrame(records)



Compute revenue with some errors


In [ ]:
df["revenue"] = df["quantity"] * df["unit_price"]


Inject some missing revenues


In [ ]:
df.loc[np.random.choice(df.index, 8, replace=False), "revenue"] = np.nan


Inject some extreme outliers


In [ ]:
df.loc[np.random.choice(df.index, 3, replace=False), "revenue"] = 9999999



Inject duplicate sale_ids


In [ ]:
df.loc[195:197, "sale_id"] = [10, 20, 30]

print("=== DATASET WITH QUALITY ISSUES ===")
print(df.head(10))
print(f"\nShape: {df.shape}")




══════════════════════════════════════════════════════════════
 CONCEPT 1 — DATA QUALITY DIMENSIONS
══════════════════════════════════════════════════════════════
The four core data quality dimensions explain WHAT can go wrong with data:

1. COMPLETENESS:
   Are all required values present?
   Measure: % of non-null values per required column
   Bad data: NULL, empty string, "N/A", -9999 (missing value placeholders)

2. ACCURACY:
   Are values within valid ranges and correct domains?
   Measure: % of values within expected bounds or matching an allowed set
   Bad data: negative quantities, future dates, unknown category codes

3. CONSISTENCY:
   Does the data agree with itself?
   Measure: % of records where computed values match stored values
   Bad data: revenue != quantity * unit_price, duplicate primary keys

4. TIMELINESS:
   Is the data fresh enough to be useful?
   Measure: time since most recent record vs expected freshness threshold
   Bad data: a "daily" table that has not been updated in 3 days


EXAMPLE ──────────────────────────────────────────────────────


In [ ]:

print("\n=== DATA QUALITY DIMENSIONS ===")
dimensions = [
    ("Completeness", "Are all required fields populated?",
     "NULL/missing values in required columns"),
    ("Accuracy",     "Are values in valid ranges and domains?",
     "Negative prices, unknown region codes, future dates"),
    ("Consistency",  "Does data agree with itself?",
     "revenue != qty * price, duplicate IDs"),
    ("Timeliness",   "Is the data current enough?",
     "Table not updated within expected window"),
]
for dim, question, example_issue in dimensions:
    print(f"\n  {dim}")
    print(f"    Question: {question}")
    print(f"    Example issue: {example_issue}")




══════════════════════════════════════════════════════════════
 EXERCISE 1
══════════════════════════════════════════════════════════════
Write a function called check_completeness(df, required_cols) that:
  1. For each column in required_cols:
     - Count null values
     - Count empty string values (for string columns)
     - Compute completeness_pct = (non-missing / total) * 100
  2. Returns a dict: {col_name: {"missing": n, "completeness_pct": x}}

Call it with df and required_cols = ["sale_id", "rep_name", "region", "revenue"]
Print each column's completeness percentage.

Expected output:
  sale_id    : 200/200 (100.0% complete)
  rep_name   : ~190/200 (~95.0% complete, 10 missing)
  region     : ~194/200 (~97.0% complete, 6 missing)
  revenue    : ~192/200 (~96.0% complete, 8 missing)


══════════════════════════════════════════════════════════════
 CONCEPT 2 — WRITING QUALITY CHECK FUNCTIONS
══════════════════════════════════════════════════════════════
Each quality check should be a standalone function that:
  - Accepts a DataFrame (and optional config parameters)
  - Returns a check_result dict with: passed (bool), failing_count, details
  - Does NOT modify the DataFrame

This design allows you to run any combination of checks and aggregate results.


EXAMPLE ──────────────────────────────────────────────────────


In [ ]:

def check_null_pct(df, column, max_null_pct=5.0):
    """Check that null percentage in a column does not exceed threshold."""
    null_count = df[column].isnull().sum()
    null_pct   = null_count / len(df) * 100
    passed     = null_pct <= max_null_pct
    return {
        "check":          f"null_pct_{column}",
        "passed":         passed,
        "failing_count":  int(null_count),
        "metric":         round(null_pct, 2),
        "threshold":      max_null_pct,
        "detail":         f"{null_pct:.1f}% nulls (limit: {max_null_pct}%)",
    }

def check_value_range(df, column, min_val=None, max_val=None):
    """Check that all numeric values fall within [min_val, max_val]."""
    col = pd.to_numeric(df[column], errors="coerce")
    mask = pd.Series([True] * len(df), index=df.index)
    if min_val is not None:
        mask &= (col >= min_val)
    if max_val is not None:
        mask &= (col <= max_val)
    failing = (~mask & col.notna()).sum()
    return {
        "check":         f"range_{column}",
        "passed":        failing == 0,
        "failing_count": int(failing),
        "detail":        f"{failing} values outside [{min_val}, {max_val}]",
    }

def check_unique(df, column):
    """Check that a column has no duplicate values."""
    dup_count = df[column].duplicated().sum()
    return {
        "check":         f"unique_{column}",
        "passed":        dup_count == 0,
        "failing_count": int(dup_count),
        "detail":        f"{dup_count} duplicate values in '{column}'",
    }

def check_allowed_values(df, column, allowed_set):
    """Check that all non-null values in a column are in the allowed set."""
    non_null   = df[column].dropna()
    invalid    = non_null[~non_null.isin(allowed_set)]
    return {
        "check":         f"allowed_values_{column}",
        "passed":        len(invalid) == 0,
        "failing_count": len(invalid),
        "detail":        f"{len(invalid)} values not in allowed set. Examples: {invalid.unique()[:3].tolist()}",
    }

def check_consistency(df, result_col, factor_a, factor_b, tolerance=0.01):
    """Check that result_col = factor_a * factor_b within tolerance."""
    expected  = df[factor_a] * df[factor_b]
    actual    = pd.to_numeric(df[result_col], errors="coerce")
    diff      = (actual - expected).abs()
    failing   = (diff > tolerance).sum()
    return {
        "check":         f"consistency_{result_col}={factor_a}*{factor_b}",
        "passed":        failing == 0,
        "failing_count": int(failing),
        "detail":        f"{failing} rows where {result_col} != {factor_a} * {factor_b}",
    }



Run individual checks


In [ ]:
print("\n=== INDIVIDUAL QUALITY CHECKS ===")

results = [
    check_null_pct(df, "rep_name",  max_null_pct=2.0),
    check_null_pct(df, "revenue",   max_null_pct=1.0),
    check_value_range(df, "quantity",  min_val=0),
    check_value_range(df, "unit_price", min_val=1.0, max_val=10000.0),
    check_value_range(df, "revenue",   max_val=500000.0),
    check_unique(df, "sale_id"),
    check_allowed_values(df, "region", {"West", "East", "Central"}),
    check_consistency(df, "revenue", "quantity", "unit_price"),
]

for r in results:
    status = "PASS" if r["passed"] else "FAIL"
    print(f"  [{status}] {r['check']:<50} | {r['detail']}")




══════════════════════════════════════════════════════════════
 EXERCISE 2
══════════════════════════════════════════════════════════════
Write two more check functions and add them to the results list.

Function 1: check_timeliness(df, date_column, max_days_old=2)
  - Parses the date column
  - Finds the most recent date in the column
  - Checks that it is no more than max_days_old days before today
  - Returns the standard check_result dict
  - For this exercise, use today = datetime(2024, 7, 20)
    (the data ends around day 200 = July 18, 2024)

Function 2: check_no_future_dates(df, date_column, reference_date)
  - Parses the date column
  - Counts how many dates are AFTER reference_date
  - Returns the standard check_result dict
  - Use reference_date = datetime(2024, 7, 20)

Run both and print the results.

Expected output:
  [PASS or FAIL] timeliness_sale_date ...
  [PASS] no_future_dates_sale_date ...


══════════════════════════════════════════════════════════════
 CONCEPT 3 — BUILDING A QUALITY REPORT
══════════════════════════════════════════════════════════════
A quality report aggregates all checks into a single summary dict.
It is produced at the end of every pipeline run and stored in a log.

The report includes:
  - Overall pass/fail status
  - Per-check results
  - Key metrics (total checks, passed, failed, overall score)


EXAMPLE ──────────────────────────────────────────────────────


In [ ]:

def run_quality_checks(df, checks_config):
    """
    Run a list of check functions and produce a quality report.
    checks_config: list of (function, kwargs) tuples
    Returns a report dict.
    """
    report = {
        "run_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "row_count":     len(df),
        "checks":        [],
        "total_checks":  0,
        "passed":        0,
        "failed":        0,
        "quality_score": 0.0,
        "overall_status": "pending",
    }

    for check_func, kwargs in checks_config:
        result = check_func(df, **kwargs)
        report["checks"].append(result)
        report["total_checks"] += 1
        if result["passed"]:
            report["passed"] += 1
        else:
            report["failed"] += 1

    total = report["total_checks"]
    report["quality_score"] = round(report["passed"] / total * 100, 1) if total > 0 else 0.0
    report["overall_status"] = "PASS" if report["failed"] == 0 else "FAIL"

    return report



Define all checks to run


In [ ]:
checks_config = [
    (check_null_pct,       {"column": "rep_name",    "max_null_pct": 10.0}),
    (check_null_pct,       {"column": "revenue",     "max_null_pct": 5.0}),
    (check_value_range,    {"column": "quantity",    "min_val": 0}),
    (check_value_range,    {"column": "unit_price",  "min_val": 1.0, "max_val": 10000.0}),
    (check_value_range,    {"column": "revenue",     "max_val": 500000.0}),
    (check_unique,         {"column": "sale_id"}),
    (check_allowed_values, {"column": "region", "allowed_set": {"West", "East", "Central"}}),
    (check_consistency,    {"result_col": "revenue", "factor_a": "quantity", "factor_b": "unit_price"}),
]

report = run_quality_checks(df, checks_config)

print("\n=== QUALITY REPORT ===")
print(f"Run timestamp:  {report['run_timestamp']}")
print(f"Row count:      {report['row_count']}")
print(f"Total checks:   {report['total_checks']}")
print(f"Passed:         {report['passed']}")
print(f"Failed:         {report['failed']}")
print(f"Quality score:  {report['quality_score']}%")
print(f"Overall status: {report['overall_status']}")

print("\nCheck details:")
for check in report["checks"]:
    status = "PASS" if check["passed"] else "FAIL"
    print(f"  [{status}] {check['check']:<50} | {check['detail']}")




══════════════════════════════════════════════════════════════
 EXERCISE 3
══════════════════════════════════════════════════════════════
A quality gate is a threshold: if quality score falls below it,
the pipeline should STOP rather than load bad data.

Write a function called quality_gate(report, min_score=90.0, critical_checks=None)
where:
  - min_score: minimum quality score percentage to pass the gate
  - critical_checks: list of check names that MUST pass regardless of score

The function should:
  1. Check if report["quality_score"] >= min_score
  2. Check if all critical_checks are in the PASSED set
  3. Return {"gate_passed": bool, "reason": str}
     If gate fails, reason explains WHY it failed.

Test with:
  quality_gate(report, min_score=70.0,
               critical_checks=["unique_sale_id", "range_quantity"])

Then test with a stricter gate:
  quality_gate(report, min_score=99.0)  # should fail

Expected output (first call):
  Gate result: {'gate_passed': True/False, 'reason': '...'}

Expected output (second call):
  Gate result: {'gate_passed': False, 'reason': 'Quality score X% below minimum 99.0%'}


══════════════════════════════════════════════════════════════
 CONCEPT 4 — SMART SCHEMA AGENT: STAGE 3 — SAFETY LAYER
══════════════════════════════════════════════════════════════
 CUMULATIVE PROJECT — Stage 3 builds on Stages 1-2 (W6-W7).
 The agent now checks every query against safety rules before running it.
 Dangerous patterns are rejected and logged as BLOCKED.

 Why this matters: a data agent that can run any SQL is a risk.
 DROP TABLE, DELETE without WHERE, and TRUNCATE can destroy data silently.
 The safety layer is the last line of defense before the database.

 New this week:
   - QuerySafetyError: a custom exception for blocked queries
   - DANGEROUS_PATTERNS: a list of (regex, reason) pairs
   - _check_safety(sql): raises QuerySafetyError if any pattern matches
   - execute_query now calls _check_safety before touching the database

 EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
import sqlite3
import re
from datetime import datetime


class QuerySafetyError(Exception):
    pass


class SmartSchemaAgent:
    """
    A governed database agent. Grows each week (W6-W11).
    Stage 3 — safety layer blocks destructive queries before execution.
    """

    DANGEROUS_PATTERNS = [
        (r"\bDROP\b",                          "DROP statement not allowed"),
        (r"\bTRUNCATE\b",                      "TRUNCATE not allowed"),
        (r"\bDELETE\s+FROM\b(?!.*\bWHERE\b)", "DELETE without WHERE"),
        (r"\bUPDATE\b(?!.*\bWHERE\b)",       "UPDATE without WHERE"),
    ]

    def __init__(self, conn: sqlite3.Connection):
        self.conn = conn
        self.cur  = conn.cursor()
        self._init_audit_log()

    def _init_audit_log(self):
        self.cur.execute('''
            CREATE TABLE IF NOT EXISTS audit_log (
                log_id    INTEGER PRIMARY KEY AUTOINCREMENT,
                timestamp TEXT NOT NULL,
                sql_text  TEXT NOT NULL,
                row_count INTEGER,
                status    TEXT NOT NULL
            )
        ''')
        self.conn.commit()

    def _check_safety(self, sql: str):
        sql_up = sql.upper()
        for pattern, reason in self.DANGEROUS_PATTERNS:
            if re.search(pattern, sql_up):
                raise QuerySafetyError(reason)

    def _write_log(self, ts, sql, row_count, status):
        self.cur.execute(
            "INSERT INTO audit_log (timestamp, sql_text, row_count, status) VALUES (?,?,?,?)",
            (ts, sql[:200], row_count, status),
        )
        self.conn.commit()

    def execute_query(self, sql: str, params: tuple = ()) -> list:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        try:
            self._check_safety(sql)
            self.cur.execute(sql, params)
            rows = self.cur.fetchall()
            self.conn.commit()
            self._write_log(ts, sql, len(rows), "OK")
            return rows
        except QuerySafetyError as e:
            self._write_log(ts, sql, 0, f"BLOCKED: {e}")
            raise
        except Exception as e:
            self._write_log(ts, sql, 0, f"ERROR: {e}")
            raise


# Demo
conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.execute("CREATE TABLE employees (id INTEGER PRIMARY KEY, name TEXT, salary REAL)")
cur.executemany("INSERT INTO employees VALUES (?,?,?)", [
    (1, "Alice", 95000), (2, "Bob", 72000), (3, "Carol", 81000)
])
conn.commit()

agent = SmartSchemaAgent(conn)

# Safe — passes through
result = agent.execute_query("SELECT name FROM employees WHERE salary > ?", (80000,))
print("High earners:", result)

# Dangerous — blocked
for dangerous_sql in ["DROP TABLE employees", "DELETE FROM employees"]:
    try:
        agent.execute_query(dangerous_sql)
    except QuerySafetyError as e:
        print(f"BLOCKED: {e}")

# Audit shows both OK and BLOCKED entries
cur2 = conn.cursor()
cur2.execute("SELECT status, sql_text FROM audit_log ORDER BY log_id")
for status, sql in cur2.fetchall():
    print(f"  {status:<35} | {sql[:40]}")

conn.close()

══════════════════════════════════════════════════════════════
 EXERCISE 4 — SMART SCHEMA AGENT: STAGE 3
══════════════════════════════════════════════════════════════
 Build the SmartSchemaAgent with the safety layer using the starting data below.
 Add one extra pattern to DANGEROUS_PATTERNS that does NOT appear in the example:

   (r"\bINSERT\s+INTO\b.*\bSELECT\b", "INSERT...SELECT not allowed — use ETL pipeline instead")

 Run these 5 queries using agent.execute_query() and handle exceptions:
   1. SELECT * FROM transactions
   2. UPDATE transactions SET amount = 9999.0 WHERE transaction_id = 1   (safe — has WHERE)
   3. DELETE FROM transactions   (no WHERE — blocked)
   4. DROP TABLE transactions    (blocked)
   5. INSERT INTO transactions SELECT * FROM transactions  (blocked by your new rule)

 After all queries, read audit_log directly and print each entry.

 Expected output:
     Transactions: [(1, ...), (2, ...), (3, ...), (4, ...)]
     Updated 1 row
     BLOCKED: DELETE without WHERE
     BLOCKED: DROP statement not allowed
     BLOCKED: INSERT...SELECT not allowed — use ETL pipeline instead
     Audit log:
       [1] OK                                 | SELECT * FROM transactions
       [2] OK                                 | UPDATE transactions SET amount...
       [3] BLOCKED: DELETE without WHERE      | DELETE FROM transactions
       [4] BLOCKED: DROP statement not allowed| DROP TABLE transactions
       [5] BLOCKED: INSERT...SELECT...        | INSERT INTO transactions SELECT...

 --- starting data ---

In [ ]:
import sqlite3
import re
from datetime import datetime

conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.execute('''
    CREATE TABLE transactions (
        transaction_id INTEGER PRIMARY KEY,
        date TEXT, amount REAL, category TEXT
    )
''')
cur.executemany("INSERT INTO transactions VALUES (?,?,?,?)", [
    (1, "2024-01-10", 1500.0, "Sales"),
    (2, "2024-01-11",  250.0, "Refund"),
    (3, "2024-01-12", 3200.0, "Sales"),
    (4, "2024-01-13",  175.0, "Operations"),
])
conn.commit()

══════════════════════════════════════════════════════════════
 CONCEPT 5 — REGRESSION GATES: PROTECTING CUMULATIVE CONTRACTS
══════════════════════════════════════════════════════════════

 The Smart Schema Agent grows across W6-W12. Each new stage builds on all
 previous ones. Without regression tests, a change to Stage 5 can silently
 break Stage 2's audit log contract — and you will not know until production.

 A regression gate is a test runner that executes ALL prior-stage contracts
 whenever any stage changes. If one breaks, the change is blocked.

 This is the direct answer to "vibe coding" breakage:
   - You update the NL->SQL (Stage 5)
   - The gate automatically runs Stage 1 through Stage 4
   - Stage 2's audit log fails because you accidentally dropped the table
   - The gate blocks the change before it reaches main

 Pattern:
   1. Write a contract function per stage — returns True/False
   2. Register all contracts in a list
   3. Run all contracts before any commit or deploy
   4. Fail fast on the first contract that breaks

EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
import sqlite3
import re
from datetime import datetime


# ── Minimal Agent definition (Stages 1-2) used only in this EXAMPLE ──────────

class _ExampleAgent:
    DANGEROUS_PATTERNS = [
        (r"\bDROP\b",                          "DROP not allowed"),
        (r"\bDELETE\s+FROM\b(?!.*\bWHERE\b)", "DELETE without WHERE"),
    ]

    def __init__(self, conn):
        self.conn = conn
        self.cur  = conn.cursor()
        self.cur.execute('''
            CREATE TABLE IF NOT EXISTS audit_log (
                log_id INTEGER PRIMARY KEY AUTOINCREMENT,
                ts TEXT, sql_text TEXT, status TEXT
            )
        ''')
        conn.commit()

    def schema(self):
        self.cur.execute("SELECT name, sql FROM sqlite_master WHERE type='table'")
        return {row[0]: row[1] for row in self.cur.fetchall()}

    def _check_safety(self, sql):
        for pat, reason in self.DANGEROUS_PATTERNS:
            if re.search(pat, sql.upper()):
                raise ValueError(reason)

    def run(self, sql, params=()):
        ts = datetime.now().strftime("%H:%M:%S")
        try:
            self._check_safety(sql)
            self.cur.execute(sql, params)
            rows = self.cur.fetchall()
            self.conn.commit()
            self.cur.execute(
                "INSERT INTO audit_log (ts, sql_text, status) VALUES (?,?,?)",
                (ts, sql[:100], "OK"),
            )
            self.conn.commit()
            return rows
        except ValueError as e:
            self.cur.execute(
                "INSERT INTO audit_log (ts, sql_text, status) VALUES (?,?,?)",
                (ts, sql[:100], f"BLOCKED: {e}"),
            )
            self.conn.commit()
            raise


# ── Contract functions: one per Agent stage ───────────────────────────────────

def stage1_schema_contract(agent) -> bool:
    """Stage 1 (W6): schema() must return a dict with at least one table."""
    schema = agent.schema()
    assert isinstance(schema, dict),  "schema() must return a dict"
    assert len(schema) >= 1,          "schema() must find at least one table"
    return True


def stage2_audit_contract(agent) -> bool:
    """Stage 2 (W7): every query must be written to audit_log."""
    before = len(agent.run("SELECT * FROM audit_log"))
    agent.run("SELECT 1")
    after_rows = agent.run("SELECT * FROM audit_log")
    assert len(after_rows) > before,  "audit_log must grow after each query"
    return True


def stage3_safety_contract(agent) -> bool:
    """Stage 3 (W8): DROP and DELETE-without-WHERE must be blocked."""
    for dangerous in ["DROP TABLE audit_log", "DELETE FROM audit_log"]:
        blocked = False
        try:
            agent.run(dangerous)
        except ValueError:
            blocked = True
        assert blocked, f"Safety layer must block: {dangerous}"
    return True


# ── Regression runner ─────────────────────────────────────────────────────────

def run_regression_gate(agent, contracts: list) -> dict:
    """Run all contract functions. Return a summary dict."""
    results = {}
    for fn in contracts:
        name = fn.__name__
        try:
            fn(agent)
            results[name] = "PASS"
        except AssertionError as e:
            results[name] = f"FAIL: {e}"
    all_pass = all(v == "PASS" for v in results.values())
    return {"all_pass": all_pass, "results": results}


# ── Run the gate ──────────────────────────────────────────────────────────────

conn_reg = sqlite3.connect(":memory:")
cur_reg  = conn_reg.cursor()
cur_reg.execute("CREATE TABLE employees (id INTEGER PRIMARY KEY, name TEXT)")
cur_reg.executemany("INSERT INTO employees VALUES (?,?)", [(1,"Alice"),(2,"Bob")])
conn_reg.commit()

agent_reg = _ExampleAgent(conn_reg)
report = run_regression_gate(agent_reg, [
    stage1_schema_contract,
    stage2_audit_contract,
    stage3_safety_contract,
])

print("=== REGRESSION GATE REPORT ===")
print(f"Overall: {'PASS' if report['all_pass'] else 'FAIL'}")
for stage, status in report["results"].items():
    print(f"  {stage:<35} : {status}")

conn_reg.close()

══════════════════════════════════════════════════════════════
 EXERCISE 5
══════════════════════════════════════════════════════════════

 The starting data below provides:
   - BrokenAgent: an agent where the safety layer has a bug — it only
     blocks DROP but silently allows DELETE without WHERE
   - The same three contract functions from the EXAMPLE above
   - run_regression_gate from the EXAMPLE above

 Do these three things:

 1. Run run_regression_gate with BrokenAgent against all 3 contracts.
    Print the report. stage3_safety_contract must FAIL.

 2. Assert that report["all_pass"] is False.

 3. Assert that report["results"]["stage3_safety_contract"] starts with "FAIL".

 The last line must print: "Regression gate correctly caught the broken safety layer."

 Expected output:
     === REGRESSION GATE REPORT ===
     Overall: FAIL
       stage1_schema_contract    : PASS
       stage2_audit_contract     : PASS
       stage3_safety_contract    : FAIL: Safety layer must block: DELETE FROM audit_log
     Regression gate correctly caught the broken safety layer.

 --- starting data ---

In [ ]:
import sqlite3
import re
from datetime import datetime


class BrokenAgent:
    """
    An agent with a bug in the safety layer: only blocks DROP, not DELETE-without-WHERE.
    This simulates what happens when someone modifies Stage 3 carelessly.
    """

    def __init__(self, conn):
        self.conn = conn
        self.cur  = conn.cursor()
        self.cur.execute('''
            CREATE TABLE IF NOT EXISTS audit_log (
                log_id INTEGER PRIMARY KEY AUTOINCREMENT,
                ts TEXT, sql_text TEXT, status TEXT
            )
        ''')
        conn.commit()

    def schema(self):
        self.cur.execute("SELECT name, sql FROM sqlite_master WHERE type='table'")
        return {row[0]: row[1] for row in self.cur.fetchall()}

    def _check_safety(self, sql):
        # BUG: only checks for DROP, misses DELETE without WHERE
        if re.search(r"\bDROP\b", sql.upper()):
            raise ValueError("DROP not allowed")

    def run(self, sql, params=()):
        ts = datetime.now().strftime("%H:%M:%S")
        try:
            self._check_safety(sql)
            self.cur.execute(sql, params)
            rows = self.cur.fetchall()
            self.conn.commit()
            self.cur.execute(
                "INSERT INTO audit_log (ts, sql_text, status) VALUES (?,?,?)",
                (ts, sql[:100], "OK"),
            )
            self.conn.commit()
            return rows
        except ValueError as e:
            self.cur.execute(
                "INSERT INTO audit_log (ts, sql_text, status) VALUES (?,?,?)",
                (ts, sql[:100], f"BLOCKED: {e}"),
            )
            self.conn.commit()
            raise


conn_broken = sqlite3.connect(":memory:")
cur_broken  = conn_broken.cursor()
cur_broken.execute("CREATE TABLE orders (id INTEGER PRIMARY KEY, amount REAL)")
cur_broken.executemany("INSERT INTO orders VALUES (?,?)", [(1, 100.0), (2, 200.0)])
conn_broken.commit()

broken_agent = BrokenAgent(conn_broken)




